In [25]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from tqdm import tqdm
tqdm.pandas()
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [26]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
submission_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

In [ ]:
import pandas as pd
import numpy as np
import torch
from dataclasses import dataclass
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
    PreTrainedTokenizerBase
)
from transformers.utils import PaddingStrategy
from datasets import Dataset
from typing import Optional, Union

@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if "label" in features[0].keys() else "labels"
        labels = [feature.pop(label_name) for feature in features]
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)] for feature in features
        ]
        flattened_features = sum(flattened_features, [])
        
        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )
        
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        batch["labels"] = torch.tensor(labels, dtype=torch.int64)
        return batch

def run_pipeline(train_df: pd.DataFrame, test_df: pd.DataFrame, model_name: str = "microsoft/deberta-v3-base") -> pd.DataFrame:
    options = ['A', 'B', 'C', 'D', 'E']
    option_to_index = {opt: i for i, opt in enumerate(options)}
    index_to_option = {i: opt for i, opt in enumerate(options)}

    train_df = train_df.copy()
    test_df = test_df.copy()

    train_df['label'] = train_df['answer'].map(option_to_index)
    test_df['label'] = 0 

    train_ds = Dataset.from_pandas(train_df)
    test_ds = Dataset.from_pandas(test_df)

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

    def preprocess_function(examples):
        first_sentences = [[context] * 5 for context in examples["prompt"]]
        second_sentences = [[examples[opt][i] for opt in options] for i in range(len(examples["prompt"]))]
        
        first_sentences = sum(first_sentences, [])
        second_sentences = sum(second_sentences, [])
        
        tokenized_examples = tokenizer(
            first_sentences,
            second_sentences,
            truncation=True,
            max_length=256,
            padding=False
        )
        return {k: [v[i : i + 5] for i in range(0, len(v), 5)] for k, v in tokenized_examples.items()}

    train_cols_to_remove = [c for c in train_ds.column_names if c != 'label']
    tokenized_train = train_ds.map(preprocess_function, batched=True, remove_columns=train_cols_to_remove)

    test_cols_to_remove = [c for c in test_ds.column_names if c != 'label']
    tokenized_test = test_ds.map(preprocess_function, batched=True, remove_columns=test_cols_to_remove)

    model = AutoModelForMultipleChoice.from_pretrained(model_name)

    # DeBERTa-v3 specific stability fix: Prefer bf16 if supported, otherwise disable mixed precision.
    bf16_supported = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

    training_args = TrainingArguments(
        output_dir="./results",
        eval_strategy="no",
        save_strategy="no",
        learning_rate=2e-5,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=8,
        num_train_epochs=3,
        weight_decay=0.01,
        fp16=False,
        bf16=bf16_supported,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        processing_class=tokenizer,
        data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer)
    )

    trainer.train()

    predictions = trainer.predict(tokenized_test).predictions
    
    top_3_indices = np.argsort(predictions, axis=1)[:, ::-1][:, :3]
    
    top_3_predictions = []
    for indices in top_3_indices:
        pred_str = " ".join([index_to_option[idx] for idx in indices])
        top_3_predictions.append(pred_str)

    submission_df = pd.DataFrame({
        'id': test_df['id'],
        'Prediction': top_3_predictions
    })

    return submission_df

if __name__ == "__main__":
    # Example instantiation logic
    # train_data = pd.read_csv("train.csv")
    # test_data = pd.read_csv("test.csv")
    # submission = run_pipeline(train_data, test_data, model_name="microsoft/deberta-v3-base")
    # submission.to_csv("submission.csv", index=False)
    pass

submission = run_pipeline(train_df, test_df, model_name="microsoft/deberta-v3-base")
submission.to_csv("submission.csv", index=False)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                

Step,Training Loss
